In [7]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np

In [8]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [9]:
resNet52_model = keras.applications.ResNet152(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling=None,
)

model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        resNet52_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.4),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet152 (Functional)          │ (None, 1, 1, 2048)     │    58,370,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 20)             │         5,140 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,559,572 (227.20 MB)

 Trainable params: 59,406,612 (226.62 MB)

 Non-trainable params: 152,960 (597.50 KB)

In [10]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 165s 259ms/step - accuracy: 0.1638 - loss: 2.8595 - val_accuracy: 0.0580 - val_loss: 3.9238
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 24s 85ms/step - accuracy: 0.4331 - loss: 1.8895 - val_accuracy: 0.1100 - val_loss: 3.1699
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 24s 85ms/step - accuracy: 0.5720 - loss: 1.4543 - val_accuracy: 0.0950 - val_loss: 8.2198
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 24s 85ms/step - accuracy: 0.4352 - loss: 1.9064 - val_accuracy: 0.3640 - val_loss: 2.3793
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 24s 86ms/step - accuracy: 0.5505 - loss: 1.5071 - val_accuracy: 0.4570 - val_loss: 2.1136


In [11]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - accuracy: 0.4772 - loss: 2.2971


[2.439239025115967, 0.4584999978542328]